# Experimentos da Dissertação — IDS com Generalização Cross-Dataset e Otimização Multiobjetivo

Pipeline de execução dos **Algoritmos 1 (NSGA-II)** e **2 (AvaliarFitness)** do Capítulo 4, com baselines do Capítulo 5.

**Regras de operação deste notebook:**
1. A **Célula 1 (Setup)** roda SEMPRE que a sessão (re)conectar — o Colab zera bibliotecas e `/content` a cada sessão.
2. Tudo que importa é gravado no **Drive** (`mestrado/Resultados`): cache de avaliações, progresso por geração e resultados finais. Nada crítico fica em `/content`.
3. **Se o Colab cair no meio de uma execução: rode a Célula 1 e depois a MESMA célula que caiu.** O cache no Drive faz a execução retomar do ponto da falha (avaliações já feitas não são recomputadas).

Repositório: https://github.com/Wadsonsp/multiobjective-ids-generalization

## 1. Setup da sessão (rodar SEMPRE ao abrir/reconectar)

In [ ]:
# === Setup: Drive + código + dependências (idempotente) ===
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('/content/multiobjective-ids-generalization'):
    !git clone https://github.com/Wadsonsp/multiobjective-ids-generalization.git
%cd /content/multiobjective-ids-generalization
!git pull
!pip install -q -r requirements.txt

# Caminhos permanentes no Drive
DRIVE_DATASETS   = '/content/drive/MyDrive/mestrado/Datasets'
DRIVE_RESULTADOS = '/content/drive/MyDrive/mestrado/Resultados'
DRIVE_CHECKPOINT = DRIVE_RESULTADOS + '/checkpoints'
os.makedirs(DRIVE_CHECKPOINT, exist_ok=True)
print('Setup concluído.')

## 2. Preparação dos dados (roda rápido se já estiver pronto)

Verifica a integridade dos CSVs contra as contagens oficiais e gera a **amostra estratificada de 2,0 milhões de fluxos** do NF-ToN-IoT-v2 (Seção 5.1) — a etapa pesada só executa na primeira vez.

In [ ]:
import pandas as pd, numpy as np

origem  = f'{DRIVE_DATASETS}/NF-ToN-IoT-v2.csv'
destino = f'{DRIVE_DATASETS}/NF-ToN-IoT-v2-2M.csv'
N_ALVO, TOTAL_OFICIAL = 2_000_000, 16_940_496

if os.path.exists(destino):
    print('Amostra de 2M já existe no Drive — nada a fazer.')
else:
    # Integridade: contagem total lendo só a coluna Attack
    total = sum(len(c) for c in pd.read_csv(origem, usecols=['Attack'], chunksize=2_000_000))
    print(f'Fluxos: {total:,} | oficial: {TOTAL_OFICIAL:,} | '
          f"{'INTEGRO' if total == TOTAL_OFICIAL else 'DIVERGENTE - investigar antes de seguir!'}")

    # Amostra estratificada em blocos (cabe na RAM do Colab gratuito)
    frac, rng, primeiro = N_ALVO / total, np.random.RandomState(42), True
    for chunk in pd.read_csv(origem, chunksize=1_000_000, low_memory=False):
        amostra = chunk.groupby('Attack', group_keys=False).apply(
            lambda g: g.sample(frac=min(1.0, frac), random_state=rng))
        amostra.to_csv(destino, mode='w' if primeiro else 'a', header=primeiro, index=False)
        primeiro = False
    n = sum(len(c) for c in pd.read_csv(destino, usecols=['Attack'], chunksize=2_000_000))
    print(f'Amostra salva: {n:,} fluxos')

In [ ]:
# Aponta o config para a amostra de 2M (sed é idempotente aqui)
!sed -i 's/arquivo: "NF-ToN-IoT-v2.csv"/arquivo: "NF-ToN-IoT-v2-2M.csv"/' src/config.yaml
!sed -i 's/amostra_estratificada: 2000000/amostra_estratificada: null/' src/config.yaml
!grep -B1 -A4 'NF-ToN-IoT-v2:' src/config.yaml

## 3. Baseline — máscara cheia (Algoritmo 2, todos os atributos)

Referência de F1 intra/cross, tempo de inferência e k(x)=d contra a qual o P* será comparado.

In [ ]:
!python src/algoritmo2_avaliacao.py --mascara cheia --amostra 50000
!cp -r Resultados/metricas/. {DRIVE_RESULTADOS}/metricas/ 2>/dev/null || (mkdir -p {DRIVE_RESULTADOS}/metricas && cp -r Resultados/metricas/. {DRIVE_RESULTADOS}/metricas/)

## 4. Baseline — RFE (Capítulo 5, monobjetivo)

Seleção 37→30 com Random Forest como estimador de referência, avaliada pelo MESMO Algoritmo 2. Use `--todos` para os 8 classificadores (Tabela 3) quando quiser a comparação completa.

In [ ]:
!python src/baseline_rfe.py --amostra 100000 --amostra-rfe 100000
!cp -r Resultados/metricas/. {DRIVE_RESULTADOS}/metricas/

## 5. Piloto do NSGA-II (medir o custo antes da execução longa)

Execução pequena (12×5) para extrapolar o tempo: o custo cresce ~linearmente com N_pop × N_gen e com o tamanho da amostra de busca. **Anote a duração impressa ao final.**

In [ ]:
!python src/algoritmo1_otimizacao.py --n-pop 12 --n-gen 5 --amostra-busca 50000 \
    --checkpoint-dir {DRIVE_CHECKPOINT}

## 6. Otimização completa (Algoritmo 1) — com retomada automática

Parâmetros definitivos calibrados pelo piloto. **Se a sessão cair: rode a Célula 1 e depois esta mesma célula** — o `--checkpoint-dir` no Drive garante que as avaliações já feitas viram cache hit e a execução continua do ponto da falha. O arquivo `progresso_*.json` no Drive sempre contém a fronteira parcial da última geração concluída.

In [ ]:
N_POP, N_GEN, AMOSTRA = 24, 15, 50000   # calibrar com o piloto da Célula 5

!python src/algoritmo1_otimizacao.py --n-pop {N_POP} --n-gen {N_GEN} \
    --amostra-busca {AMOSTRA} --checkpoint-dir {DRIVE_CHECKPOINT}

# Resultados finais (P* + histórico) para o Drive
!mkdir -p {DRIVE_RESULTADOS}/pareto {DRIVE_RESULTADOS}/metricas
!cp -r Resultados/pareto/.   {DRIVE_RESULTADOS}/pareto/
!cp -r Resultados/metricas/. {DRIVE_RESULTADOS}/metricas/

## 7. Inspeção do P* e do progresso

In [ ]:
import json, glob

# Fronteira parcial (atualizada a cada geração — útil DURANTE/após quedas)
for p in sorted(glob.glob(f'{DRIVE_CHECKPOINT}/progresso_*.json')):
    with open(p) as f:
        estado = json.load(f)
    print(f"{os.path.basename(p)} -> geração {estado['geracao']}, "
          f"{estado['avaliacoes']} avaliações, "
          f"{len(estado['mascaras_parciais'])} soluções na fronteira parcial")

# Último P* final salvo
paretos = sorted(glob.glob(f'{DRIVE_RESULTADOS}/pareto/pareto_*.json'))
if paretos:
    with open(paretos[-1]) as f:
        pareto = json.load(f)
    print(f'\nÚltimo P*: {os.path.basename(paretos[-1])} '
          f'({len(pareto["mascaras"])} soluções)')
    for i, (obj, attrs) in enumerate(zip(pareto['objetivos'],
                                         pareto['atributos_selecionados_por_solucao'])):
        print(f'  sol {i}: 1-F1_intra={obj[0]:.4f} | 1-F1_cross={obj[1]:.4f} | '
              f'custo={obj[2]:.4f} | k={len(attrs)}')

## 8. Avaliação final das soluções escolhidas (dados completos)

A busca usou subamostra; os números **da dissertação** vêm desta reavaliação com `--amostra 0`. Troque o índice de `--solucao` para cada solução de interesse (maior F1 cross, menor k, joelho da fronteira).

In [ ]:
ARQUIVO_PARETO = sorted(glob.glob('Resultados/pareto/pareto_*.json'))[-1]
SOLUCAO = 0

!python src/algoritmo2_avaliacao.py --pareto {ARQUIVO_PARETO} --solucao {SOLUCAO} --amostra 0
!cp -r Resultados/metricas/. {DRIVE_RESULTADOS}/metricas/